# Monte Carlo MLE and Posterior Analysis

This notebook demonstrates `MCMLE` and `MCPosterior` — the two Monte Carlo
analysis classes in `src/mc/`. Both classes:

1. Repeatedly draw a mock observation from `MockGenerator`.
2. Fit the observation with either maximum-likelihood estimation or MCMC
   posterior sampling.
3. Record results to a resumable parquet file.

Running the analysis cells a second time **resumes** from the saved file rather
than starting from scratch.

**Diagnostics computed**

| Analysis | Output |
|----------|--------|
| `MCMLE` | bias, scatter of point estimates |
| `MCPosterior` | bias/scatter of posterior means, typical uncertainty size, empirical coverage |

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

project_root = next(
    (
        candidate
        for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (candidate / "src" / "mc" / "mle.py").exists()
    ),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from numcosmo_py import Ncm

from src.mock.mset import create_mock_mset
from src.likelihood.mset import create_mset
from src.mock.generator import MockGenerator
from src.mock.source import HamanaGalaxySource
from src.cli.builders import build_likelihood_factory, LikelihoodType
from src.mc.mle import MCMLE
from src.mc.posterior import MCPosterior
from src.utils.utils import CoordSystem

__name__ = "NcContext"
Ncm.cfg_init()
Ncm.cfg_set_log_handler(lambda msg: None)

In [ ]:
custom_params = {
    "figure.dpi": 300,
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "legend.title_fontsize": 9,
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": False,
    "ytick.right": False,
}
sns.set_theme(context="paper", style="ticks", palette="colorblind", rc=custom_params)

## 1. Parameters

In [ ]:
# ── True cluster ─────────────────────────────────────────────────────────────
TRUE_LOG10M = 14.8   # log10(M_200c / M_sun)
Z_CLUSTER   = 0.3
RA          = 0.0    # degrees
DEC         = 0.0

# ── Sampling geometry ────────────────────────────────────────────────────────
R_MIN    = 0.3       # Mpc/h
R_MAX    = 3.0       # Mpc/h
N_GALS   = 3000       # galaxies per mock realisation
H0       = 70.0
h        = H0 / 100.0

# ── Likelihood ───────────────────────────────────────────────────────────────
FPARAMS        = ["NcHaloMassSummary:log10MDelta"]
FPARAMS_BOUNDS = [(13.0, 16.0)]
BIN_EDGES      = np.logspace(np.log10(R_MIN / h), np.log10(R_MAX / h), 9).tolist()

# ── MC iterations ────────────────────────────────────────────────────────────
N_ITER_MLE  = 15     # number of successful MLE iterations
N_ITER_POST = 15     # number of successful posterior iterations
SEED        = 42

print(f"Bin edges (Mpc): {[f'{e:.3f}' for e in BIN_EDGES]}")

## 2. Galaxy source

`HamanaGalaxySource` loads photometric-redshift galaxy catalogues from the
Hamana et al. YAML + `.gvar` file format.  It samples galaxies from clusters
whose redshift is within `z_tol` of the target cluster redshift, weighting
by inverse Δz.

In [ ]:
CLUSTERS_PATH = Path("/home/caio/Development/cosmology/clusters")

source = HamanaGalaxySource.from_directory(CLUSTERS_PATH)

cluster_zs = [c.z_cluster for c in source._clusters]
print(f"Loaded {len(source._clusters)} Hamana clusters")
print(f"Cluster redshifts: {[f'{z:.3f}' for z in cluster_zs]}")

## 3. Mock generator and likelihood factory

Two separate model sets are required:
- `mock_mset` — owned by the generator; holds the **true** cluster mass and is
  never modified by the fitter.
- `fit_mset` — owned by the likelihood; the optimizer updates its parameters
  during each fit.

In [ ]:
mock_mset = create_mock_mset(
    H0=H0, ra=RA, dec=DEC, z=Z_CLUSTER, log10MDelta=TRUE_LOG10M,
)
fit_mset = create_mset(
    H0=H0, ra=RA, dec=DEC, z=Z_CLUSTER, log10MDelta=TRUE_LOG10M,
)

generator = MockGenerator(
    mset=mock_mset,
    source=source,
    true_ra=RA,
    true_dec=DEC,
    r_min=R_MIN,
    r_max=R_MAX,
    r_miscenter=0.0,
    n_gals=N_GALS,
    p_cut=0.98,
    delta_z=0.2,
)

lik_factory = build_likelihood_factory(
    fit_mset,
    likelihood_type=LikelihoodType.BINNED_SIGMA,
    coord_system=CoordSystem.CELESTIAL,
    fparams=FPARAMS,
    fparams_bounds=FPARAMS_BOUNDS,
    bin_edges=BIN_EDGES,
    radius_bounds=None,
)

print("Generator and likelihood factory ready.")

## 4. Monte Carlo MLE analysis

`MCMLE` runs `n_iter` successful Nelder–Mead optimisations and writes results
to `output`. Re-running this cell resumes from the saved parquet file.

In [ ]:
MLE_OUTPUT = Path("mc_mle_results.parquet")

mc_mle = MCMLE(
    generator=generator,
    likelihood_factory=lik_factory,
    n_iter=N_ITER_MLE,
    output=MLE_OUTPUT,
    true_params={"NcHaloMassSummary:log10MDelta": TRUE_LOG10M},
    seed=SEED,
    fparams_bounds=dict(zip(FPARAMS, FPARAMS_BOUNDS)),
)

mle_df = mc_mle.run()

In [ ]:
mle_result = mc_mle.summary()
print(f"\nMCMLE summary  ({mle_result.n_successful}/{mle_result.n_total} successful)")
print("-" * 55)
for param, stats in mle_result.per_param.items():
    param_short = param.split(":", 1)[-1]
    print(f"  {param_short}")
    print(f"    mean   = {stats['mean']:.4f}")
    print(f"    std    = {stats['std']:.4f}")
    print(f"    median = {stats['median']:.4f}")
    if 'bias' in stats:
        print(f"    bias   = {stats['bias']:+.4f}  (mean - truth)")

In [ ]:
mle_df

In [ ]:
col = "NcHaloMassSummary:log10MDelta"
successful = mle_df[mle_df["success"] == True]
values = successful[col].to_numpy()

fig, ax = plt.subplots(figsize=(5, 3.5))

ax.hist(
    values,
    bins=max(5, len(values) // 3),
    color=sns.color_palette()[0],
    edgecolor="white",
    linewidth=0.5,
    label="MLE estimates",
)
ax.axvline(
    TRUE_LOG10M,
    color="k",
    ls="--",
    lw=1.2,
    label=rf"Truth $= {TRUE_LOG10M}$",
)
ax.axvline(
    np.mean(values),
    color=sns.color_palette()[1],
    ls="-",
    lw=1.2,
    label=rf"Mean $= {np.mean(values):.3f}$",
)

bias = np.mean(values) - TRUE_LOG10M
ax.set_xlabel(r"$\log_{10}(M_{200c}\,/\,M_\odot)$")
ax.set_ylabel("Count")
ax.set_title(
    rf"MC MLE: bias $= {bias:+.3f}$,  scatter $= {np.std(values, ddof=1):.3f}$"
)
ax.legend()
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig("mc_mle_histogram.pdf", bbox_inches="tight")
plt.show()

## 5. Monte Carlo posterior analysis

`MCPosterior` replaces the Nelder–Mead optimiser with an emcee MCMC sampler.
For each iteration it stores the posterior summary statistics
(`_mean`, `_median`, `_std`, `_q16`, `_q84`, `_q025`, `_q975`), which are then
used to compute bias, typical uncertainty size, and empirical coverage.

Adjust `nsamples` and `nwalkers` for production runs; the values below are
chosen to keep the notebook fast.

In [ ]:
POST_OUTPUT = Path("mc_posterior_results.parquet")

mc_post = MCPosterior(
    generator=generator,
    likelihood_factory=lik_factory,
    n_iter=N_ITER_POST,
    output=POST_OUTPUT,
    true_params={"NcHaloMassSummary:log10MDelta": TRUE_LOG10M},
    seed=SEED,
    nsamples=2000,
    nwalkers=4,
    nthreads=1,
    burn_in=200,
    progress=False,
    fparams_bounds=dict(zip(FPARAMS, FPARAMS_BOUNDS)),
)

post_df = mc_post.run()

In [ ]:
post_result = mc_post.summary()
print(f"\nMCPosterior summary  ({post_result.n_successful}/{post_result.n_total} successful)")
print("-" * 55)
for param, stats in post_result.per_param.items():
    print(f"  {param}")
    print(f"    bias (mean)       = {stats['bias_mean']:+.4f}")
    print(f"    bias (median)     = {stats['bias_median']:+.4f}")
    print(f"    spread (mean)     = {stats['spread_mean']:.4f}")
    print(f"    spread (median)   = {stats['spread_median']:.4f}")
    print(f"    uncertainty 1sigma = {stats['uncertainty_1sigma']:.4f}  (avg half-width)")
    print(f"    uncertainty 2sigma = {stats['uncertainty_2sigma']:.4f}")
    print(f"    coverage 68%%      = {stats['coverage_1sigma']:.2f}  (ideal: 0.68)")
    print(f"    coverage 95%%      = {stats['coverage_2sigma']:.2f}  (ideal: 0.95)")

In [ ]:
# Column names produced by MCPosterior strip the model prefix
param_col  = "log10MDelta"
successful = post_df[post_df["success"] == True]

means  = successful[f"{param_col}_mean"].to_numpy()
q16s   = successful[f"{param_col}_q16"].to_numpy()
q84s   = successful[f"{param_col}_q84"].to_numpy()
widths = (q84s - q16s) / 2.0   # half-width of 68% CI

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

# ── Left: distribution of posterior means ────────────────────────────────────
axes[0].hist(
    means,
    bins=max(4, len(means) // 2),
    color=sns.color_palette()[2],
    edgecolor="white",
    linewidth=0.5,
    label="Posterior mean",
)
axes[0].axvline(TRUE_LOG10M, color="k", ls="--", lw=1.2,
                label=rf"Truth $= {TRUE_LOG10M}$")
axes[0].axvline(np.mean(means), color=sns.color_palette()[3], ls="-", lw=1.2,
                label=rf"Mean $= {np.mean(means):.3f}$")
axes[0].set_xlabel(r"$\log_{10}(M_{200c}\,/\,M_\odot)$")
axes[0].set_ylabel("Count")
axes[0].set_title("Posterior mean estimates")
axes[0].legend(fontsize=8)
sns.despine(ax=axes[0])

# ── Right: distribution of 1sigma half-widths ────────────────────────────────
axes[1].hist(
    widths,
    bins=max(4, len(widths) // 2),
    color=sns.color_palette()[4],
    edgecolor="white",
    linewidth=0.5,
)
axes[1].axvline(np.mean(widths), color="k", ls="--", lw=1.2,
                label=rf"Mean $= {np.mean(widths):.3f}$")
axes[1].set_xlabel(r"$(q_{84} - q_{16})\,/\,2$")
axes[1].set_ylabel("Count")
axes[1].set_title(r"Posterior $1\sigma$ half-width")
axes[1].legend(fontsize=8)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig("mc_posterior_summary.pdf", bbox_inches="tight")
plt.show()

## 6. MLE vs posterior — point-estimate comparison

Both estimators should be centred near the true mass. The posterior mean
typically has slightly higher variance than the MLE (due to the finite chain
length), but comes with a calibrated uncertainty.

In [ ]:
mle_vals  = mle_df[mle_df["success"] == True]["NcHaloMassSummary:log10MDelta"].to_numpy()
post_vals = post_df[post_df["success"] == True]["log10MDelta_mean"].to_numpy()

fig, ax = plt.subplots(figsize=(5, 3.5))

bins = np.linspace(
    min(mle_vals.min(), post_vals.min()) - 0.1,
    max(mle_vals.max(), post_vals.max()) + 0.1,
    12,
)
ax.hist(mle_vals,  bins=bins, alpha=0.6, color=sns.color_palette()[0], label="MLE")
ax.hist(post_vals, bins=bins, alpha=0.6, color=sns.color_palette()[2], label="Posterior mean")
ax.axvline(TRUE_LOG10M, color="k", ls="--", lw=1.2,
           label=rf"Truth $= {TRUE_LOG10M}$")

ax.set_xlabel(r"$\log_{10}(M_{200c}\,/\,M_\odot)$")
ax.set_ylabel("Count")
ax.set_title("MLE vs posterior mean")
ax.legend()
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

print("\nPoint-estimate comparison")
print(f"  MLE:            mean = {mle_vals.mean():.4f},  std = {mle_vals.std(ddof=1):.4f},  bias = {mle_vals.mean() - TRUE_LOG10M:+.4f}")
print(f"  Posterior mean: mean = {post_vals.mean():.4f},  std = {post_vals.std(ddof=1):.4f},  bias = {post_vals.mean() - TRUE_LOG10M:+.4f}")

## CLI equivalent

The same analysis can be launched from the command line:

```bash
# MC MLE
wl-mass-fit mc-mle \
  --output mc_mle_results.parquet \
  --n-iter 15 \
  --true-ra 0.0 --true-dec 0.0 \
  --log10m 14.5 --redshift 0.3 \
  --r-min 0.3 --r-max 3.0 --r-miscenter 0.0 \
  --n-gals 300 \
  --clusters-path /home/caio/Development/cosmology/clusters \
  --likelihood binned-sigma \
  --fparams-bounds "13.0:16.0"

# MC posterior
wl-mass-fit mc-posterior \
  --output mc_posterior_results.parquet \
  --n-iter 5 \
  --true-ra 0.0 --true-dec 0.0 \
  --log10m 14.5 --redshift 0.3 \
  --r-min 0.3 --r-max 3.0 --r-miscenter 0.0 \
  --n-gals 300 \
  --clusters-path /home/caio/Development/cosmology/clusters \
  --likelihood binned-sigma \
  --fparams-bounds "13.0:16.0" \
  --nsamples 500 --nwalkers 8 --burn-in 100
```